# Initial-margin CVaR MKABSDE

This notebook displays the retained call and butterfly campaigns. It reads the generated JSON files and contains no solver implementation.

## Complete retraining command

Run from the repository root:

```bash
python -m initial_margin.run_campaign \
  --product all \
  --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 \
  --degree 3 \
  --margin-period 0.02 \
  --device cuda
```

The paper defaults use ten independent seeds, 24 Gauss--Legendre nodes for the call tail, and 64 nodes for the complete butterfly law.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')

## Control-variate prices and numerical variability

The direct column targets the anticipated equation. The non-linear and linear columns target the two proxy BSDEs. Each entry is the mean plus or minus the sample standard deviation across seeds.

In [2]:
article_prices = {'call': (2.1959, 2.1971), 'butterfly': (0.3112, 0.3112)}
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    nonlinear_article, linear_article = article_prices[product]
    lines = [
        f'### ATM {product}',
        '',
        '| $N$ | direct MKABSDE | non-linear proxy | linear proxy |',
        '|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {row['direct_mkabsde_control_variate_price_mean']:.6f} ± {row['direct_mkabsde_control_variate_price_std']:.2e} | "
            f"{row['nonlinear_proxy_control_variate_price_mean']:.6f} ± {row['nonlinear_proxy_control_variate_price_std']:.2e} | "
            f"{row['linear_proxy_control_variate_price_mean']:.6f} ± {row['linear_proxy_control_variate_price_std']:.2e} |"
        )
    lines.append(f'| Article | — | {nonlinear_article:.6f} | {linear_article:.6f} |')
    display(Markdown('\n'.join(lines)))

### ATM call

| $N$ | direct MKABSDE | non-linear proxy | linear proxy |
|---:|---:|---:|---:|
| 50 | 2.192020 ± 6.15e-05 | 2.196516 ± 7.28e-05 | 2.196009 ± 2.86e-05 |
| 100 | 2.191944 ± 6.05e-05 | 2.196443 ± 7.52e-05 | 2.195946 ± 2.02e-05 |
| 200 | 2.191923 ± 5.96e-05 | 2.196415 ± 6.00e-05 | 2.195920 ± 2.47e-05 |
| Article | — | 2.195900 | 2.197100 |

### ATM butterfly

| $N$ | direct MKABSDE | non-linear proxy | linear proxy |
|---:|---:|---:|---:|
| 50 | 0.310359 ± 2.76e-06 | 0.310245 ± 3.17e-06 | 0.311271 ± 2.03e-06 |
| 100 | 0.310360 ± 4.46e-06 | 0.310237 ± 5.13e-06 | 0.311294 ± 1.63e-06 |
| 200 | 0.310353 ± 5.22e-06 | 0.310230 ± 5.02e-06 | 0.311303 ± 6.41e-07 |
| Article | — | 0.311200 | 0.311200 |

## Interpretation

The proxy equations are no longer anticipated, so the backward implementation reduces to the classical implicit BSDE scheme with Hermite conditional regressions. The direct MKABSDE has no independent oracle in the source article. The reported sample standard deviations measure the combined Monte Carlo and regression variability; they are not a pure regression-error estimator. The three retained time grids show stability, but their small and non-monotone adjacent differences do not support a robust empirical $O(h)$ claim.